# Part 4 — Dynamic Scheduler Analysis

For each run this notebook produces a three-panel figure with shared x-axis:
- Top Gantt chart: which colour-coded job runs on which core at each moment in time.
- Middle: QPS left y-axis and p95 read latency with the 0.8 ms SLO line (right y-axis).
- Bottom: per-core CPU utilization (%) from the `cpu_N.csv` logged by the controller.

A combined p95-only overlay across all runs follows.

Section index
- [Q1 - Memcached T/C characterisation](#q1)
- [Q3 - Dynamic scheduler (15 s intervals)](#q3)
- [Q4 - Dynamic scheduler (5 s intervals)](#q4)

In [ ]:
import re
import csv
import io
import glob
import os
from datetime import datetime, timezone
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.ticker as ticker
import numpy as np
import pandas as pd

# Experiment parameters
Q1_DIR   = Path("../data/part-4-q1")
Q3_DIR   = Path("../data/part-4")
Q4_DIR   = Path("../data/part-4-q4")
RUNS     = [1, 2, 3]

Q3_INTERVAL_S = 15
Q4_INTERVAL_S = 5

SLO_MS = 0.8
SLO_US = SLO_MS * 1000

TOTAL_CORES = 4

In [ ]:
# Color map: colors match the report template
JOB_COLORS = {
    "memcached":     "#1f77b4",   # not in template; keep a neutral blue
    "barnes":        "#AACCCA",
    "blackscholes":  "#CCA000",
    "canneal":       "#CCCCAA",
    "freqmine":      "#0CCA00",
    "radix":         "#00CCA0",
    "streamcluster": "#CCACCA",
    "vips":          "#CC0A00",
}
JOBS_ORDER = list(JOB_COLORS.keys())

In [ ]:
# Parsers (Q3/Q4)

def parse_mcperf(path: Path):
    """Return (unix_start_s, list of per-interval dicts with p95, qps, target)."""
    unix_start = None
    intervals = []
    with open(path) as fh:
        for line in fh:
            line = line.strip()
            if line.startswith("Timestamp start:"):
                unix_start = int(line.split(":", 1)[1].strip()) / 1_000
            elif line.startswith("read"):
                cols = line.split()
                intervals.append({
                    "p95":    float(cols[12]),
                    "qps":    float(cols[16]),
                    "target": float(cols[17]),
                })
    return unix_start, intervals


def parse_core_list(token: str) -> list:
    return [int(x) for x in re.findall(r"\d+", token)]


def parse_jobs(path: Path):
    """Return (scheduler_start_dt, events). Timestamps tagged UTC."""
    scheduler_start = None
    events = []
    with open(path) as fh:
        for line in fh:
            parts = line.strip().split()
            if not parts:
                continue
            ts    = datetime.fromisoformat(parts[0]).replace(tzinfo=timezone.utc)
            etype = parts[1]
            job   = parts[2] if len(parts) > 2 else None
            if job == "scheduler":
                if etype == "start":
                    scheduler_start = ts
                continue
            cores = parse_core_list(parts[3]) if len(parts) > 3 else []
            if etype == "end":
                cores = []
            events.append({"t": ts, "type": etype, "job": job, "cores": cores})
    return scheduler_start, events


def build_gantt(scheduler_start, events, t_end):
    state = {}
    segments = []

    def flush(job, until_s):
        if job not in state or not state[job]["cores"]:
            return
        t0 = state[job]["t_start"]
        dur = until_s - t0
        if dur > 0:
            for core in state[job]["cores"]:
                segments.append((job, core, t0, dur))

    for ev in sorted(events, key=lambda e: e["t"]):
        job = ev["job"]
        t_s = (ev["t"] - scheduler_start).total_seconds()
        flush(job, t_s)
        if ev["type"] == "end":
            state.pop(job, None)
        else:
            state[job] = {"cores": ev["cores"], "t_start": t_s}
    for job in list(state.keys()):
        flush(job, t_end)
    return segments


def parse_cpu_log(path: Path, scheduler_start):
    times = []
    cores = [[], [], [], []]
    try:
        with open(path, newline="") as fh:
            reader = csv.DictReader(fh)
            for row in reader:
                ts  = datetime.fromisoformat(row["timestamp"]).replace(tzinfo=timezone.utc)
                t_s = (ts - scheduler_start).total_seconds()
                times.append(t_s)
                for i in range(4):
                    key = f"core{i}"
                    cores[i].append(float(row[key]) if key in row else 0.0)
    except FileNotFoundError:
        pass
    return times, cores

In [ ]:
# Three-panel plot function (Q3 / Q4)

def plot_run(data_dir: Path, run: int, interval_s: int, label: str = ""):
    mcperf_path = data_dir / f"run_{run}" / f"mcperf_{run}.txt"
    jobs_path   = data_dir / f"run_{run}" / f"jobs_{run}.txt"
    cpu_path    = data_dir / f"run_{run}" / f"cpu_{run}.csv"

    unix_start, intervals = parse_mcperf(mcperf_path)
    sched_start, events   = parse_jobs(jobs_path)

    # Fallback: if jobs log is missing or empty, use mcperf unix_start as t=0
    has_jobs = sched_start is not None
    if not has_jobs:
        sched_start = datetime.fromtimestamp(unix_start, tz=timezone.utc)

    mcperf_offset = unix_start - sched_start.timestamp()
    t_mid   = [mcperf_offset + (i + 0.5) * interval_s for i in range(len(intervals))]
    t_end_s = mcperf_offset + len(intervals) * interval_s

    p95_us = [iv["p95"] for iv in intervals]
    qps    = [iv["qps"]  for iv in intervals]

    gantt_segs          = build_gantt(sched_start, events, t_end_s)
    cpu_times, cpu_data = parse_cpu_log(cpu_path, sched_start)
    has_cpu             = len(cpu_times) > 0

    fig, axes = plt.subplots(
        3, 1, figsize=(15, 9), sharex=True,
        gridspec_kw={"height_ratios": [1.5, 2, 1.5], "hspace": 0.05},
    )
    fig.suptitle(f"Part 4 — {label}Run {run}  (interval={interval_s}s)",
                 fontsize=13, fontweight="bold")
    ax_gantt, ax_lat, ax_cpu = axes

    # Gantt
    for (job, core, t0, dur) in gantt_segs:
        ax_gantt.broken_barh([(t0, dur)], (core - 0.4, 0.8),
                             facecolors=JOB_COLORS.get(job, "black"),
                             edgecolors="white", linewidth=0.3)
    ax_gantt.set_yticks(range(TOTAL_CORES))
    ax_gantt.set_yticklabels([f"Core {c}" for c in range(TOTAL_CORES)], fontsize=9)
    ax_gantt.set_ylabel("CPU core", fontsize=10)
    ax_gantt.set_ylim(-0.6, TOTAL_CORES - 0.4)
    ax_gantt.grid(axis="x", ls=":", alpha=0.4)
    if not has_jobs:
        ax_gantt.text(0.5, 0.5, "jobs log not available for this run",
                      ha="center", va="center", transform=ax_gantt.transAxes,
                      fontsize=11, color="grey", style="italic")
    else:
        seen = {seg[0] for seg in gantt_segs}
        ax_gantt.legend(
            handles=[mpatches.Patch(color=JOB_COLORS[j], label=j)
                     for j in JOBS_ORDER if j in seen],
            fontsize=8, loc="upper right", ncol=4, framealpha=0.8)

    # QPS + p95
    c_qps, c_lat = "#7f7f7f", "#1f77b4"
    ax_lat.fill_between(t_mid, [q / 1000 for q in qps], alpha=0.15, color=c_qps, step="mid")
    ax_lat.step(t_mid, [q / 1000 for q in qps], where="mid", color=c_qps, lw=1.0, label="QPS (K)")
    ax_lat.set_ylabel("QPS (K)", fontsize=10, color=c_qps)
    ax_lat.tick_params(axis="y", labelcolor=c_qps)
    ax_lat.set_ylim(bottom=0)
    ax_lat.grid(axis="y", ls=":", alpha=0.3)

    ax_p95 = ax_lat.twinx()
    ax_p95.plot(t_mid, p95_us, marker="o", ms=2.5, lw=1.2, color=c_lat, label="p95 latency")
    ax_p95.axhline(SLO_US, color="red", lw=1.5, ls="--", label=f"SLO ({SLO_MS} ms)")
    violations = [(t, p) for t, p in zip(t_mid, p95_us) if p > SLO_US]
    if violations:
        ax_p95.scatter(*zip(*violations), color="red", s=40, zorder=5,
                       label=f"Violation ({len(violations)}x)")
    ax_p95.set_ylabel("p95 latency (µs)", fontsize=10, color=c_lat)
    ax_p95.tick_params(axis="y", labelcolor=c_lat)
    ax_p95.set_ylim(bottom=0)
    l1, lb1 = ax_lat.get_legend_handles_labels()
    l2, lb2 = ax_p95.get_legend_handles_labels()
    ax_p95.legend(l1 + l2, lb1 + lb2, fontsize=9, loc="upper right")

    # Per-core CPU
    core_colors = ["#1f77b4", "#ff7f0e", "#2ca02c", "#d62728"]
    if has_cpu:
        for i in range(TOTAL_CORES):
            ax_cpu.plot(cpu_times, cpu_data[i], lw=0.8,
                        color=core_colors[i], alpha=0.85, label=f"Core {i}")
        ax_cpu.legend(fontsize=8, loc="upper right", ncol=2)
    else:
        ax_cpu.text(0.5, 0.5, "cpu_N.csv not available",
                    ha="center", va="center", transform=ax_cpu.transAxes,
                    fontsize=11, color="grey")
    ax_cpu.set_ylabel("CPU util (% / core)", fontsize=10)
    x_label = "Time from scheduler start (s)" if has_jobs else "Time from mcperf start (s)"
    ax_cpu.set_xlabel(x_label, fontsize=10)
    ax_cpu.set_ylim(-2, 105)
    ax_cpu.axhline(100, color="black", lw=0.6, ls=":")
    ax_cpu.grid(axis="y", ls=":", alpha=0.4)

    plt.savefig(data_dir / f"run_{run}" / f"part4_run{run}.pdf", bbox_inches="tight")
    plt.show()

    n_viol, n_int = len(violations), len(intervals)
    print(f"Run {run}: {n_viol}/{n_int} SLO violations ({100*n_viol/n_int:.1f}%)")
    print(f"  max p95 = {max(p95_us):.1f} µs  |  mean p95 = {np.mean(p95_us):.1f} µs")
    if not has_jobs:
        print(f"  (jobs log was empty — Gantt not shown; x-axis is relative to mcperf start)")
    print()

<a id='q1'></a>
## Q1 - Memcached T/C Configuration Sweep (5 K–125 K QPS)

Data collected with `make start-part-4-1` with results in `data/part-4-q1/`.
- Q1a: 9 configs (T in {1,2,3} * C in {1,2,3}), 3 runs each - latency vs. QPS.
- Q1d: T chosen from Q1c where C in {1,2,3}, 1 run - latency left y-axis + CPU util right y-axis per C.

In [ ]:
# Q1 helpers
import io as _io

def parse_mcperf_scan(filepath):
    """Parse a --scan mcperf log; return DataFrame with QPS, p95_ms columns."""
    with open(filepath) as f:
        lines = f.readlines()
    table = [l for l in lines if l.startswith('#') or l.startswith('read')]
    df = pd.read_csv(_io.StringIO(''.join(table)), sep=r'\s+')
    df.columns = [c.replace('#', '') for c in df.columns]
    df['p95_ms'] = df['p95'] / 1000.0
    return df[['target', 'QPS', 'p95_ms']]


def parse_pidstat_cpu(filepath):
    """
    Parse pidstat -u output; return list of (sample_index, cpu_pct) tuples.
    The %CPU column reports total CPU across all threads (can exceed 100%).
    """
    readings = []
    try:
        with open(filepath) as f:
            for line in f:
                parts = line.split()
                # Data lines: timestamp  UID  PID  %usr  %system  %guest  %wait  %CPU  CPU  Command
                # Skip header/blank/summary lines
                if len(parts) < 9:
                    continue
                if parts[-1] not in ('memcached', ):
                    continue
                try:
                    cpu_pct = float(parts[7])   # %CPU column
                    readings.append(cpu_pct)
                except (ValueError, IndexError):
                    pass
    except FileNotFoundError:
        pass
    return readings


# Load Q1a data
Q1A_CONFIGS   = [(T, C) for T in [1,2,3] for C in [1,2,3]]
Q1A_RUNS      = [1, 2, 3]
Q1A_SLO_MS    = SLO_MS

q1a_rows = []
for T, C in Q1A_CONFIGS:
    for run in Q1A_RUNS:
        p = Q1_DIR / "q1a" / f"t{T}_c{C}_run{run}.txt"
        if not p.exists():
            continue
        df = parse_mcperf_scan(p)
        df['T'] = T; df['C'] = C; df['run'] = run
        q1a_rows.append(df)

if q1a_rows:
    q1a_df = pd.concat(q1a_rows, ignore_index=True)
    q1a_stats = q1a_df.groupby(['T','C','target']).agg(
        qps_mean=('QPS','mean'), qps_std=('QPS','std'),
        p95_mean=('p95_ms','mean'), p95_std=('p95_ms','std'),
        n_runs=('run','count'),
    ).reset_index()
    print(f"Q1a: {len(q1a_rows)} files loaded, "
          f"{q1a_df['target'].nunique()} QPS steps, "
          f"{q1a_df.groupby(['T','C']).ngroups} T/C configs")
else:
    q1a_stats = None
    print("Q1a data not yet available - run: make start-part-4-1")

In [ ]:
# Q1a Plot: p95 latency vs. QPS for all 9 T/C configurations
if q1a_stats is None:
    print("No Q1a data - skipping plot.")
else:
    import matplotlib as mpl
    cmap    = mpl.colormaps['tab10']
    markers = ['o','s','D','^','v','p','*','X','P']
    # Line style distinguishes C; colour distinguishes T
    T_colors   = {1: cmap(0), 2: cmap(1), 3: cmap(2)}
    C_linestyle = {1: '-', 2: '--', 3: ':'}

    fig, ax = plt.subplots(figsize=(13, 7))

    for idx, (T, C) in enumerate(Q1A_CONFIGS):
        sub = q1a_stats[(q1a_stats['T']==T) & (q1a_stats['C']==C)].sort_values('target')
        if sub.empty:
            continue
        n = int(sub['n_runs'].iloc[0])
        ax.errorbar(
            sub['qps_mean'] / 1000, sub['p95_mean'],
            xerr=sub['qps_std'] / 1000, yerr=sub['p95_std'],
            label=f"T={T}, C={C} (n={n})",
            marker=markers[idx], markersize=7,
            markeredgecolor='black', markeredgewidth=0.5,
            capsize=3, linestyle=C_linestyle[C],
            linewidth=1.4, color=T_colors[T],
        )

    ax.axhline(Q1A_SLO_MS, color='red', lw=1.8, ls='--',
               label=f'SLO ({Q1A_SLO_MS} ms p95)')
    ax.text(124, Q1A_SLO_MS + 0.04, f'SLO = {Q1A_SLO_MS} ms',
            color='red', ha='right', fontsize=9)

    ax.set_title('Q1a - Memcached p95 Latency vs. Throughput (T threads, C cores)',
                 fontsize=13, fontweight='bold')
    ax.set_xlabel('Achieved Throughput (K QPS)', fontsize=12, fontweight='bold')
    ax.set_ylabel('p95 Latency (ms)', fontsize=12, fontweight='bold')
    ax.xaxis.set_major_formatter(plt.FuncFormatter(
        lambda x, _: '0' if x == 0 else f'{x:.0f}K'))
    ax.set_xlim(0, 130)
    ax.set_xticks(range(0, 131, 10))
    ax.set_ylim(0, 7)
    ax.grid(True, which='both', linestyle='--', alpha=0.5)
    ax.legend(title='Config (colour=T, style=C)', loc='upper left',
              fontsize=9, ncol=3, frameon=True)
    fig.text(0.5, -0.01, 'Error bars: ±1 SD across 3 runs.',
             ha='center', fontsize=9, color='grey')
    plt.tight_layout()
    plt.savefig(Q1_DIR / 'q1a_latency_vs_qps.pdf', bbox_inches='tight')
    plt.show()

    # Max sustainable QPS table
    print(f"\n{'Config':<12} {'Max QPS within SLO':>22} {'p95 there (ms)':>16} {'SLO first violated at':>22}")
    print('-' * 76)
    for T, C in Q1A_CONFIGS:
        sub = q1a_stats[(q1a_stats['T']==T) & (q1a_stats['C']==C)].sort_values('target')
        within   = sub[sub['p95_mean'] <= Q1A_SLO_MS]
        violated = sub[sub['p95_mean'] >  Q1A_SLO_MS]
        max_qps  = f"{within['qps_mean'].iloc[-1]/1000:.1f}K" if not within.empty else "0"
        p95_at   = f"{within['p95_mean'].iloc[-1]:.3f}"       if not within.empty else "N/A"
        first_viol = (f"{violated['target'].iloc[0]/1000:.0f}K"
                      if not violated.empty else ">125K")
        print(f"T={T}, C={C}       {max_qps:>22} {p95_at:>16} {first_viol:>22}")

### Q1d — Latency + CPU Utilization:

all T values per core count.

The script collects data for all T in {1,2,3} at each C,
so we can show empirically why a particular T is the right choice for Q1c,
rather than just picking the maximum.

For each C the graph overlays three latency curves, one per T,
and three CPU-utilization curves, making two things visible:
- which T values respect the 0.8 ms SLO across the full QPS range.
- whether adding more cores actually helps, i.e. does CPU scale with C.

The final subsection reproduces the single-T graph required by the handout.

In [ ]:
# Q1d parsers

def parse_pidstat_cpu(filepath):
    """
    Parse pidstat -u output; return list of %CPU readings for the memcached
    process.  %CPU can exceed 100 % for multi-threaded processes (it sums
    across all hardware threads the process is using).
    """
    readings = []
    try:
        with open(filepath) as f:
            for line in f:
                parts = line.split()
                # pidstat data lines end with the command name ('memcached')
                # Format: timestamp  UID  PID  %usr  %system  %guest  %wait  %CPU  CPU  Command
                if len(parts) >= 10 and parts[-1] == 'memcached':
                    try:
                        readings.append(float(parts[7]))   # %CPU column
                    except ValueError:
                        pass
    except FileNotFoundError:
        pass
    return readings


def load_q1d_config(T, C):
    """Return (qps_arr, p95_arr, cpu_arr) for a Q1d (T, C) run, or None."""
    mp = Q1_DIR / "q1d" / f"t{T}_c{C}_mcperf.txt"
    cp = Q1_DIR / "q1d" / f"t{T}_c{C}_cpu.txt"
    if not mp.exists():
        return None
    df = parse_mcperf_scan(mp)
    qps    = df['QPS'].values / 1000          # K QPS
    p95_ms = df['p95_ms'].values
    n      = len(p95_ms)

    cpu_raw = parse_pidstat_cpu(cp)
    # Align: keep last n readings (pidstat may have started 1 interval early)
    if len(cpu_raw) > n:
        cpu_raw = cpu_raw[-n:]
    cpu = np.array(cpu_raw + [np.nan] * max(0, n - len(cpu_raw)))
    return qps, p95_ms, cpu


# Q1d: overlay all T values, one figure per C
import matplotlib as mpl_mod

T_COLORS  = {1: mpl_mod.colormaps['tab10'](0),
             2: mpl_mod.colormaps['tab10'](1),
             3: mpl_mod.colormaps['tab10'](2)}
T_MARKERS = {1: 'o', 2: 's', 3: 'D'}

any_q1d = any(
    (Q1_DIR / "q1d" / f"t{T}_c{C}_mcperf.txt").exists()
    for T in [1,2,3] for C in [1,2,3]
)

if not any_q1d:
    print("Q1d data not yet available - run: make start-part-4-1")
else:
    for C in [1, 2, 3]:
        fig, ax_lat = plt.subplots(figsize=(12, 5))
        ax_cpu = ax_lat.twinx()

        plotted = False
        for T in [1, 2, 3]:
            result = load_q1d_config(T, C)
            if result is None:
                print(f"  Missing: T={T} C={C}")
                continue
            qps, p95_ms, cpu = result
            col = T_COLORS[T]
            mk  = T_MARKERS[T]

            # p95 latency (left axis, solid line)
            ax_lat.plot(qps, p95_ms, marker=mk, ms=5, lw=1.5,
                        color=col, label=f"p95  T={T}")
            # CPU utilisation (right axis, dashed line)
            ax_cpu.plot(qps, cpu, marker=mk, ms=4, lw=1.0,
                        color=col, ls='--', alpha=0.7, label=f"CPU  T={T}")
            plotted = True

        if not plotted:
            plt.close()
            continue

        # SLO line
        ax_lat.axhline(SLO_MS, color='red', lw=1.8, ls='--', label=f'SLO ({SLO_MS} ms)')

        cpu_max_pct = C * 100
        ax_cpu.axhline(cpu_max_pct, color='grey', lw=0.8, ls=':', alpha=0.6)
        ax_cpu.set_ylim(0, cpu_max_pct * 1.15)
        ax_cpu.set_ylabel(f'CPU Utilisation (0–{cpu_max_pct}%)', fontsize=11)

        ax_lat.set_xlabel('Achieved Throughput (K QPS)', fontsize=11, fontweight='bold')
        ax_lat.set_ylabel('p95 Latency (ms)', fontsize=11)
        ax_lat.set_xlim(0, 130)
        ax_lat.set_ylim(0)
        ax_lat.xaxis.set_major_formatter(plt.FuncFormatter(
            lambda x, _: '0' if x == 0 else f'{x:.0f}K'))
        ax_lat.grid(True, linestyle='--', alpha=0.4)

        # Merged legend: latency entries first, then CPU, then SLO
        l1, lb1 = ax_lat.get_legend_handles_labels()
        l2, lb2 = ax_cpu.get_legend_handles_labels()
        ax_lat.legend(l1 + l2, lb1 + lb2, fontsize=9, loc='upper left',
                      ncol=2, frameon=True)

        plt.title(f'Q1d - C={C} core(s): p95 Latency and CPU util for T=1,2,3',
                  fontsize=12, fontweight='bold')
        plt.tight_layout()
        plt.savefig(Q1_DIR / f"q1d_c{C}_all_threads.pdf", bbox_inches='tight')
        plt.show()

        # Print SLO summary for this C
        print(f"C={C} - max QPS within SLO per thread count:")
        for T in [1, 2, 3]:
            result = load_q1d_config(T, C)
            if result is None: continue
            qps, p95_ms, cpu = result
            ok = qps[p95_ms <= SLO_MS]
            max_ok = f"{ok.max():.0f}K" if len(ok) else "none"
            max_cpu = f"{np.nanmax(cpu):.1f}%" if len(cpu) else "N/A"
            print(f"  T={T}: max QPS within SLO = {max_ok:>5}   peak CPU = {max_cpu}")
        print()

In [ ]:
# Q1d final: three separate graphs for the chosen T in handout format
# Set Q1D_CHOSEN_T to the T value you selected in Q1c based on the plots above.
Q1D_CHOSEN_T = 3

for C in [1, 2, 3]:
    result = load_q1d_config(Q1D_CHOSEN_T, C)
    if result is None:
        print(f"Q1d C={C} T={Q1D_CHOSEN_T}: data not available")
        continue
    qps, p95_ms, cpu = result

    cpu_max_pct = C * 100
    fig, ax_lat = plt.subplots(figsize=(11, 5))
    ax_cpu = ax_lat.twinx()

    ax_lat.plot(qps, p95_ms, marker='o', ms=5, lw=1.5,
                color='#1f77b4', label='p95 latency')
    ax_lat.axhline(SLO_MS, color='red', lw=1.5, ls='--', label=f'SLO ({SLO_MS} ms)')
    viol = qps[p95_ms > SLO_MS]
    if len(viol):
        ax_lat.scatter(viol, p95_ms[p95_ms > SLO_MS],
                       color='red', s=50, zorder=5,
                       label=f'Violation ({len(viol)}x)')

    ax_cpu.plot(qps, cpu, marker='s', ms=4, lw=1.2,
                color='#ff7f0e', ls='--', label=f'CPU util (max={cpu_max_pct}%)')
    ax_cpu.axhline(cpu_max_pct, color='#ff7f0e', lw=0.8, ls=':', alpha=0.7)
    ax_cpu.set_ylim(0, cpu_max_pct * 1.15)
    ax_cpu.set_ylabel(f'CPU Utilisation (0–{cpu_max_pct}%)', fontsize=11,
                      color='#ff7f0e')
    ax_cpu.tick_params(axis='y', labelcolor='#ff7f0e')

    ax_lat.set_xlabel('Achieved Throughput (K QPS)', fontsize=11, fontweight='bold')
    ax_lat.set_ylabel('p95 Latency (ms)', fontsize=11, color='#1f77b4')
    ax_lat.tick_params(axis='y', labelcolor='#1f77b4')
    ax_lat.set_xlim(0, 130)
    ax_lat.set_ylim(0)
    ax_lat.xaxis.set_major_formatter(plt.FuncFormatter(
        lambda x, _: '0' if x == 0 else f'{x:.0f}K'))
    ax_lat.grid(True, linestyle='--', alpha=0.4)

    l1, lb1 = ax_lat.get_legend_handles_labels()
    l2, lb2 = ax_cpu.get_legend_handles_labels()
    ax_lat.legend(l1 + l2, lb1 + lb2, fontsize=9, loc='upper left')

    plt.title(f'Q1d - T={Q1D_CHOSEN_T} threads, C={C} core(s)',
              fontsize=12, fontweight='bold')
    plt.tight_layout()
    plt.savefig(Q1_DIR / f"q1d_final_t{Q1D_CHOSEN_T}_c{C}.pdf", bbox_inches='tight')
    plt.show()
    print(f"C={C}: {len(viol)} SLO violation(s)  "
          f"max p95={p95_ms.max():.3f} ms  "
          f"max CPU={np.nanmax(cpu):.1f}%")
print()

<a id='q3'></a>
## Q3 — Dynamic Scheduler (15-second intervals)

In [ ]:
for run in RUNS:
    plot_run(Q3_DIR, run, Q3_INTERVAL_S, label="Q3 ")

### Q3 — Combined p95 overlay

In [ ]:
fig, ax = plt.subplots(figsize=(14, 4))
colors_runs = ["#1f77b4", "#ff7f0e", "#2ca02c"]
for run, color in zip(RUNS, colors_runs):
    unix_start, intervals = parse_mcperf(Q3_DIR / f"run_{run}" / f"mcperf_{run}.txt")
    sched_start, _        = parse_jobs(Q3_DIR / f"run_{run}" / f"jobs_{run}.txt")
    mcperf_offset = unix_start - sched_start.timestamp()
    t_mid  = [mcperf_offset + (i + 0.5) * Q3_INTERVAL_S for i in range(len(intervals))]
    p95_us = [iv["p95"] for iv in intervals]
    n_viol = sum(1 for p in p95_us if p > SLO_US)
    ax.plot(t_mid, p95_us, marker="o", ms=2.5, lw=1.2, color=color, alpha=0.85,
            label=f"Run {run}  ({n_viol} violations)")
ax.axhline(SLO_US, color="red", lw=1.5, ls="--", label=f"SLO ({SLO_MS} ms)")
ax.set_xlabel("Time from scheduler start (s)", fontsize=10)
ax.set_ylabel("p95 latency (µs)", fontsize=10)
ax.set_title("Part 4 Q3 — p95 latency across all 3 runs (interval=15 s)", fontsize=12)
ax.set_ylim(bottom=0)
ax.legend(fontsize=9)
ax.grid(axis="y", ls=":", alpha=0.5)
plt.tight_layout()
plt.savefig(Q3_DIR / "part4_q3_summary.pdf", bbox_inches="tight")
plt.show()

### Q3 — Job runtimes, makespan, and SLO violations

Per-job execution time: mean ± std across 3 runs, total makespan, and SLO violation ratio
during the batch-job execution window, from first container start to last container end.

In [ ]:
# Q3: per-job runtimes, makespan, SLO violations
import statistics

JOBS_REPORT_ORDER = [
    "streamcluster", "freqmine", "canneal", "vips",
    "blackscholes", "barnes", "radix",
]

def job_durations_and_window(events):
    """Return (durations_dict, makespan_s, first_batch_start_dt, last_batch_end_dt).

    Ignores memcached and scheduler events; uses only batch job start/end pairs.
    """
    starts = {}
    durations = {}
    first_start = None
    last_end = None
    for ev in sorted(events, key=lambda e: e["t"]):
        job = ev["job"]
        if job in ("memcached",):
            continue
        if ev["type"] == "start":
            starts[job] = ev["t"]
            if first_start is None:
                first_start = ev["t"]
        elif ev["type"] == "end" and job in starts:
            dur = (ev["t"] - starts[job]).total_seconds()
            durations[job] = dur
            last_end = ev["t"]
    makespan = (last_end - first_start).total_seconds() if first_start and last_end else None
    return durations, makespan, first_start, last_end


def slo_violations_in_window(data_dir, run, interval_s, t_start_dt, t_end_dt):
    """Count mcperf intervals with p95 > SLO_US whose midpoint falls within [t_start, t_end]."""
    unix_start, intervals = parse_mcperf(data_dir / f"run_{run}" / f"mcperf_{run}.txt")
    t0_s = t_start_dt.timestamp()
    t1_s = t_end_dt.timestamp()
    total, viols = 0, 0
    for i, iv in enumerate(intervals):
        mid = unix_start + (i + 0.5) * interval_s
        if t0_s <= mid <= t1_s:
            total += 1
            if iv["p95"] > SLO_US:
                viols += 1
    return viols, total


# Collect per-run data
all_durations = {job: [] for job in JOBS_REPORT_ORDER}
makespans = []
slo_results = []

for run in RUNS:
    jobs_path = Q3_DIR / f"run_{run}" / f"jobs_{run}.txt"
    if not jobs_path.exists():
        print(f"Run {run}: jobs_{run}.txt not found - skipping")
        continue

    _, events = parse_jobs(jobs_path)
    durs, ms, t0, t1 = job_durations_and_window(events)
    makespans.append(ms)

    for job in JOBS_REPORT_ORDER:
        all_durations[job].append(durs.get(job, float("nan")))

    viols, total = slo_violations_in_window(Q3_DIR, run, Q3_INTERVAL_S, t0, t1)
    slo_results.append((run, viols, total))
    print(f"Run {run}: makespan={ms:.1f}s | "
          f"SLO violations={viols}/{total} ({100*viols/total:.1f}%)")

# Print summary table
print()
print(f"{'Job':<16} {'Mean [s]':>10} {'Std [s]':>10}")
print("-" * 38)
for job in JOBS_REPORT_ORDER:
    vals = [v for v in all_durations[job] if not np.isnan(v)]
    mean = statistics.mean(vals) if vals else float("nan")
    std  = statistics.stdev(vals) if len(vals) > 1 else 0.0
    print(f"{job:<16} {mean:>10.1f} {std:>10.1f}")
print("-" * 38)
ms_mean = statistics.mean(makespans) if makespans else float("nan")
ms_std  = statistics.stdev(makespans) if len(makespans) > 1 else 0.0
print(f"{'total time':<16} {ms_mean:>10.1f} {ms_std:>10.1f}")

print()
print("SLO violation ratio per run (p95 > 0.8 ms, during batch window):")
for run, viols, total in slo_results:
    ratio = viols / total if total > 0 else 0.0
    print(f"  Run {run}: {viols}/{total} = {ratio:.4f} ({100*ratio:.2f}%)")

<a id='q4'></a>
## Q4 — Dynamic Scheduler (5-second intervals)

Run `make start-part-4-4 RUN=1`, `RUN=2`, `RUN=3` to collect data first.

### Q4 — Job runtimes, makespan, and SLO violations

Per-job execution time: mean ± std across 3 runs, total makespan, and SLO violation ratio
during the batch-job execution window, from first container start to last container end.
Computed identically to Q3 but with `Q4_INTERVAL_S = 5` s per mcperf measurement step.

In [ ]:
# Q4: per-job runtimes, makespan, SLO violations
# Reuses job_durations_and_window() and slo_violations_in_window() defined in the Q3 cell.

q4_all_durations = {job: [] for job in JOBS_REPORT_ORDER}
q4_makespans = []
q4_slo_results = []

for run in RUNS:
    jobs_path = Q4_DIR / f"run_{run}" / f"jobs_{run}.txt"
    if not jobs_path.exists():
        print(f"Run {run}: jobs_{run}.txt not found - skipping")
        continue

    _, events = parse_jobs(jobs_path)
    durs, ms, t0, t1 = job_durations_and_window(events)

    if t0 is None or t1 is None:
        print(f"Run {run}: jobs log empty - cannot determine batch window")
        continue

    q4_makespans.append(ms)
    for job in JOBS_REPORT_ORDER:
        q4_all_durations[job].append(durs.get(job, float("nan")))

    viols, total = slo_violations_in_window(Q4_DIR, run, Q4_INTERVAL_S, t0, t1)
    q4_slo_results.append((run, viols, total))
    print(f"Run {run}: makespan={ms:.1f}s | "
          f"SLO violations={viols}/{total} ({100*viols/total:.2f}%) | "
          f"window={t0.strftime('%H:%M:%S')}–{t1.strftime('%H:%M:%S')} ({(t1-t0).total_seconds():.0f}s)")

# Summary table
print()
print(f"{'Job':<16} {'Mean [s]':>10} {'Std [s]':>10}")
print("-" * 38)
for job in JOBS_REPORT_ORDER:
    vals = [v for v in q4_all_durations[job] if not np.isnan(v)]
    mean = statistics.mean(vals) if vals else float("nan")
    std  = statistics.stdev(vals) if len(vals) > 1 else 0.0
    print(f"{job:<16} {mean:>10.1f} {std:>10.1f}")
print("-" * 38)
ms_mean = statistics.mean(q4_makespans) if q4_makespans else float("nan")
ms_std  = statistics.stdev(q4_makespans) if len(q4_makespans) > 1 else 0.0
print(f"{'total time':<16} {ms_mean:>10.1f} {ms_std:>10.1f}")

print()
print("SLO violation ratio per run (p95 > 0.8 ms, during batch window):")
total_viols, total_pts = 0, 0
for run, viols, total in q4_slo_results:
    ratio = viols / total if total > 0 else 0.0
    print(f"  Run {run}: {viols}/{total} = {ratio:.4f} ({100*ratio:.2f}%)")
    total_viols += viols
    total_pts   += total
if total_pts > 0:
    print(f"  Overall: {total_viols}/{total_pts} = {total_viols/total_pts:.4f} "
          f"({100*total_viols/total_pts:.2f}%)")

# Comparison with Q3
print()
print("=== Q3 vs Q4 comparison ===")
print(f"{'Metric':<30} {'Q3 (15s)':>12} {'Q4 (5s)':>12}")
print("-" * 56)
q3_ms_vals = [v for v in makespans if v]  # makespans defined in Q3 cell
print(f"{'Makespan mean [s]':<30} {statistics.mean(q3_ms_vals):>12.1f} {ms_mean:>12.1f}")
print(f"{'Makespan std [s]':<30} {statistics.stdev(q3_ms_vals):>12.1f} {ms_std:>12.1f}")
q3_total_v = sum(v for _, v, _ in slo_results)   # slo_results from Q3 cell
q3_total_p = sum(t for _, _, t in slo_results)
q4_total_v = sum(v for _, v, _ in q4_slo_results)
q4_total_p = sum(t for _, _, t in q4_slo_results)
print(f"{'SLO violations (total)':<30} {q3_total_v:>10}/{q3_total_p} {q4_total_v:>10}/{q4_total_p}")

In [ ]:
for run in RUNS:
    run_dir = Q4_DIR / f"run_{run}"
    if not run_dir.exists():
        print(f"Q4 run {run}: data not yet available ({run_dir})")
        continue
    plot_run(Q4_DIR, run, Q4_INTERVAL_S, label="Q4 ")

### Q4 — Combined p95 overlay

In [ ]:
q4_runs_available = [r for r in RUNS if (Q4_DIR / f"run_{r}" / f"mcperf_{r}.txt").exists()]

if q4_runs_available:
    fig, ax = plt.subplots(figsize=(14, 4))
    colors_runs = ["#1f77b4", "#ff7f0e", "#2ca02c"]
    for run, color in zip(q4_runs_available, colors_runs):
        unix_start, intervals = parse_mcperf(Q4_DIR / f"run_{run}" / f"mcperf_{run}.txt")
        sched_start, _        = parse_jobs(Q4_DIR / f"run_{run}" / f"jobs_{run}.txt")
        if sched_start is None:
            sched_start = datetime.fromtimestamp(unix_start, tz=timezone.utc)
        mcperf_offset = unix_start - sched_start.timestamp()
        t_mid  = [mcperf_offset + (i + 0.5) * Q4_INTERVAL_S for i in range(len(intervals))]
        p95_us = [iv["p95"] for iv in intervals]
        n_viol = sum(1 for p in p95_us if p > SLO_US)
        ax.plot(t_mid, p95_us, marker="o", ms=2.5, lw=1.2, color=color, alpha=0.85,
                label=f"Run {run}  ({n_viol} violations)")
    ax.axhline(SLO_US, color="red", lw=1.5, ls="--", label=f"SLO ({SLO_MS} ms)")
    ax.set_xlabel("Time (s)", fontsize=10)
    ax.set_ylabel("p95 latency (µs)", fontsize=10)
    ax.set_title("Part 4 Q4 - p95 latency across runs (interval=5 s)", fontsize=12)
    ax.set_ylim(bottom=0)
    ax.legend(fontsize=9)
    ax.grid(axis="y", ls=":", alpha=0.5)
    plt.tight_layout()
    plt.savefig(Q4_DIR / "part4_q4_summary.pdf", bbox_inches="tight")
    plt.show()
else:
    print("No Q4 runs available yet. Run 'make start-part-4-4 RUN=1/2/3' first.")